In [1]:
!git clone https://github.com/bhujbalatharva252-sketch/Recommendation-Systems-benchmarking

Cloning into 'Recommendation-Systems-benchmarking'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 59 (delta 0), reused 2 (delta 0), pack-reused 56 (from 1)
Receiving objects: 100% (59/59), 41.38 MiB | 8.46 MiB/s, done.
Resolving deltas: 100% (15/15), done.
Updating files: 100% (32/32), done.


In [5]:
!pip install scikit-surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 31.0 MB/s eta 0:00:00


In [6]:
import pandas as pd
from surprise import SVD
from surprise import Dataset, Reader

In [10]:
%cd Recommendation-Systems-benchmarking/

/content/Recommendation-Systems-benchmarking


# Load Train data

In [11]:
train=pd.read_csv("data/processed/train.csv")


In [12]:
train =train[["UserID","MovieID","Rating"]]     # Columns required

In [24]:
movies = pd.read_csv("data/raw/movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Title", "Genres"],
    encoding="latin-1")
movies.head()

,MovieID,Title,Genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


# Convert data into surprise format

In [25]:
reader=Reader(rating_scale=(1,5))

In [26]:
data = Dataset.load_from_df(train, reader)

In [27]:
trainset=data.build_full_trainset()         # convert Ids into indexes as surprise dont directly work with pandas dataframe

# Train SVD model

In [28]:
model=SVD(random_state=42)
model.fit(trainset)

# Recommendation function

In [32]:
def recommend_svd(UserID,movies,train,model,top_n=10):
  seen_movies = set(train[train["UserID"] == UserID]["MovieID"])     # Movies already seen by user
  all_movies=train["MovieID"].unique()        # all movie IDs

  scores = []

  # predict ratings for unseen movies
  for movie in all_movies:
        if movie not in seen_movies:
            pred_rating = model.predict(UserID,movie).est
            scores.append((movie,pred_rating))

  # sort predicted rating high to low
  scores.sort(key=lambda x: x[1], reverse=True)
  top_movies = [m for m, _ in scores[:top_n]]       # top-n movies

  return movies[movies["MovieID"].isin(top_movies)][["MovieID", "Title", "Genres"]]


In [34]:
# Example
recommend=recommend_svd(1,movies,train,model)
recommend.head(10)

,MovieID,Title,Genres
315,318,"Shawshank Redemption, The (1994)",Drama
589,593,"Silence of the Lambs, The (1991)",Drama|Thriller
847,858,"Godfather, The (1972)",Action|Crime|Drama
892,904,Rear Window (1954),Mystery|Thriller
900,912,Casablanca (1942),Drama|Romance|War
1242,1262,"Great Escape, The (1963)",Adventure|War
1252,1272,Patton (1970),Drama|War
1950,2019,Seven Samurai (The Magnificent Seven) (Shichin...,Action|Drama
2836,2905,Sanjuro (1962),Action|Adventure
3400,3469,Inherit the Wind (1960),Drama
